# SC-VAE — attention + perceptual/GAN (اجرای صحت‌سنجی)

این نوت‌بوک **هر دو مولفهٔ کلیدیِ مقاله** را فعال می‌کند:
1. **attention یادگیرنده** برای وزن‌دهی α (`--attention GAT`)
2. **خطای perceptual (LPIPS) + adversarial (PatchGAN)**

اسکریپت `main-stage1-gan.py` حلقهٔ کاملِ G+D را دارد (چون `trainer_scvae.py` ناقص است)، ولی
مدل و pipelineِ داده همان کد اصلی است.

> ⚠️ این اجرا فقط برای **صحت‌سنجیِ سریع** است (دادهٔ کوچک، چند epoch). بعد از تأیید درستی،
> با پارامترهای واقعی‌تر (N و epoch بیشتر) اجرا می‌کنیم.
> ⚙️ قبل از اجرا: **Runtime → Change runtime type → GPU**.


## ۱) دریافت کد و نصب

In [ ]:
!git clone https://github.com/e-shams-d/SC-VAE.git

In [ ]:
import os
os.chdir('/content/SC-VAE')
!pip install -q omegaconf easydict tensorboardX datasets lpips piq

## ۲) دانلود خودکارِ FFHQ واقعی (کوچک برای صحت‌سنجی)

فقط `N=128` چهرهٔ واقعی را stream می‌کنیم و به train/val مجزا تقسیم می‌کنیم.


In [ ]:
import os, io
from PIL import Image as PILImage
from datasets import load_dataset

root = 'dataset/FFHQ/resized'
os.makedirs(root, exist_ok=True)

N = 128                       # کوچک، فقط برای صحت‌سنجی
ds = load_dataset("bitmind/ffhq-256", split="train", streaming=True)

names = []
for i, ex in enumerate(ds):
    if i >= N:
        break
    img = ex['image']
    if isinstance(img, dict):
        img = PILImage.open(io.BytesIO(img['bytes']))
    name = f'{i:05d}.jpg'
    img.convert('RGB').save(os.path.join(root, name))
    names.append(name)

n_val = 32
val_names, train_names = names[:n_val], names[n_val:]
open('img_datasets/assets/ffhqtrain.txt', 'w').write('\n'.join(train_names) + '\n')
open('img_datasets/assets/ffhqvalidation.txt', 'w').write('\n'.join(val_names) + '\n')
print(f'saved {len(names)} FFHQ images -> {len(train_names)} train / {len(val_names)} val')

## ۳) آموزش با attention(GAT) + perceptual + GAN

| پارامتر | مقدار | توضیح |
|---------|-------|-------|
| `--attention` | GAT | attention یادگیرنده (graph-attention) |
| `experiment.epochs` | 5 | فقط صحت‌سنجی (سریع) |
| `experiment.batch_size` | 4 | امن برای T4 |
| `optimizer.init_lr` | 2.0e-4 | + gradient clipping |

خروجی هر خط شامل `recon / sdl / pcpt / gen / disc` است؛ و هر epoch یک خطِ
`[epoch N] validation reconstruction loss:`.
🆘 `CUDA out of memory` → `experiment.batch_size=2`. اگر GAT کند بود → `--attention eq`.


In [ ]:
!python main-stage1-gan.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --attention GAT --device cuda --num-workers 2 \
  experiment.epochs=5 experiment.batch_size=4 \
  optimizer.init_lr=2.0e-4 optimizer.warmup.min_lr=1.0e-5

## ۴) نمودار همگرایی

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = sorted(glob.glob('results/logs/FFHQ/gan_*'), key=os.path.getmtime)[-1]
ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
ea.Reload()

def scal(tag):
    ev = ea.Scalars(tag)
    return [e.step for e in ev], [e.value for e in ev]

tags = ea.Tags()['scalars']
plt.figure(figsize=(8, 5))
for tag, lab in [('loss/train/reconstruction', 'train recon'),
                 ('loss/test/reconstruction', 'val recon'),
                 ('loss/train/loss_pcpt', 'perceptual'),
                 ('loss/train/loss_gen', 'generator(adv)'),
                 ('loss/train/loss_disc', 'discriminator')]:
    if tag in tags:
        s, v = scal(tag)
        plt.plot(s, v, label=lab, marker='o' if 'test' in tag else None, alpha=0.8)
plt.xlabel('training step'); plt.ylabel('loss')
plt.title('SC-VAE (attention + perceptual/GAN)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('results/convergence_gan.png', dpi=120, bbox_inches='tight')
plt.show()

## ۵) ارزیابی (PSNR / SSIM / LPIPS + بازسازی)

⚠️ حتماً `--attention GAT` را بده تا مسیرِ forwardِ ارزیابی با مدلِ آموزش‌دیده یکی باشد.


In [ ]:
import glob, os
ckpt = sorted(glob.glob('results/models/FFHQ/gan_*/best.pt'), key=os.path.getmtime)[-1]
print('using checkpoint:', ckpt)

!python evaluate.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --load-path "{ckpt}" --attention GAT --device cuda --batch-size 8

In [ ]:
from IPython.display import Image
Image('results/eval/reconstruction_compare.png')

## ۶) مرحلهٔ بعد — پارامترهای واقعی‌تر

بعد از تأیید درستیِ این اجرا، برای کیفیت بهتر این‌ها را زیاد کن (در سلول‌های بالا):
- `N = 512` (یا بیشتر) در سلول دانلود داده
- `experiment.epochs=30` (یا بیشتر) در سلول آموزش
- در صورت امکانِ حافظه، `experiment.batch_size=8`
